#03 - Data Quality Silver

### Purpose:
  1. Read standardized staging data

  2. Apply semantic Data Quality checks

  3. Measure DQ metrics

  4. Quarantine failed records

  5. Write trusted Silver Delta tables

  6. Create unified Silver client table

In [0]:
from pyspark.sql import functions as F

CATALOG = "bfsi_mdm_dev"

SYSTEMS = [
    "core",
    "crm",
    "kyc",
    "wealth",
]

MAX_PLAUSIBLE_AGE = 110

###DQ function

1. Completeness — canonical name
2. Source identity integrity
3. Phone validity
4. Email validity
5. DOB validity / age consistency
6. Date cannot be in the future
7. Raw/normalized consistency
8. Combine all checks.

In [0]:
def apply_dq_checks(df):

    checks = []

    # ========================================================
    # 1. Completeness — canonical name
    # ========================================================

    checks.append(
        F.when(
            F.col("normalized_name") == "",
            F.lit(
                "completeness: normalized_name is blank"
            )
        )
    )

    # ========================================================
    # 2. Source identity integrity
    # ========================================================

    checks.append(
        F.when(
            F.col("source_system") == "",
            F.lit(
                "integrity: source_system is blank"
            )
        )
    )

    checks.append(
        F.when(
            F.col("source_id") == "",
            F.lit(
                "integrity: source_id is blank"
            )
        )
    )

    # ========================================================
    # 3. Phone validity
    #
    # Empty phone is not automatically an error.
    # An existing malformed phone is.
    # ========================================================

    checks.append(
        F.when(
            (
                F.col("normalized_phone") != ""
            )
            &
            ~F.col(
                "normalized_phone"
            ).rlike(
                r"^\d{7,15}$"
            ),
            F.lit(
                "validity: normalized_phone is invalid"
            )
        )
    )

    # ========================================================
    # 4. Email validity
    # ========================================================

    checks.append(
        F.when(
            (
                F.col("normalized_email") != ""
            )
            &
            ~F.col(
                "normalized_email"
            ).rlike(
                r"^[^@\s]+@[^@\s]+\.[^@\s]+$"
            ),
            F.lit(
                "validity: normalized_email is invalid"
            )
        )
    )

    # ========================================================
    # 5. DOB validity / age consistency
    # ========================================================

    dob_date = F.try_to_date(
        F.col("normalized_dob"),
        F.lit("yyyy-MM-dd")
    )

    age = (
        F.datediff(
            F.current_date(),
            dob_date
        )
        / 365.25
    )

    checks.append(
        F.when(
            (
                F.col("normalized_dob") != ""
            )
            &
            (
                dob_date.isNull()
            ),
            F.lit(
                "validity: normalized_dob is invalid"
            )
        )
    )

    checks.append(
        F.when(
            (
                F.col("normalized_dob") != ""
            )
            &
            (
                age > MAX_PLAUSIBLE_AGE
            ),
            F.lit(
                "consistency: DOB implies age over 110"
            )
        )
    )

    # ========================================================
    # 6. Date cannot be in the future
    # ========================================================

    checks.append(
        F.when(
            (
                F.col("normalized_dob") != ""
            )
            &
            (
                dob_date > F.current_date()
            ),
            F.lit(
                "consistency: DOB is in the future"
            )
        )
    )

    # ========================================================
    # 7. Raw/normalized consistency
    #
    # We only check that if a raw value exists,
    # its normalized counterpart isn't unexpectedly blank.
    # ========================================================

    checks.append(
        F.when(
            (
                F.trim(
                    F.col("name")
                ) != ""
            )
            &
            (
                F.col("normalized_name") == ""
            ),
            F.lit(
                "consistency: name lost during normalization"
            )
        )
    )

    checks.append(
        F.when(
            (
                F.trim(
                    F.col("phone")
                ) != ""
            )
            &
            (
                F.col("normalized_phone") == ""
            ),
            F.lit(
                "consistency: phone lost during normalization"
            )
        )
    )

    # ========================================================
    # Combine all checks.
    # ========================================================

    return df.withColumn(
        "dq_issues",
        F.array_compact(
            F.array(*checks)
        )
    )

####Generate DQ output

In [0]:
def process_dq_for_source(system):

    print("=" * 80)
    print(f"DQ PROCESSING: {system.upper()}")
    print("=" * 80)

    df = spark.table(
        f"{CATALOG}.silver.stg_{system}_standardized"
    )

    input_count = df.count()

    print(
        f"Input rows: {input_count:,}"
    )

    # --------------------------------------------------------
    # Apply DQ
    # --------------------------------------------------------

    checked = apply_dq_checks(df)

    # --------------------------------------------------------
    # PASS
    # --------------------------------------------------------

    passed = (
        checked
        .filter(
            F.size("dq_issues") == 0
        )
        .drop("dq_issues")
    )

    # --------------------------------------------------------
    # FAIL
    # --------------------------------------------------------

    failed = (
        checked
        .filter(
            F.size("dq_issues") > 0
        )
        .withColumn(
            "dq_failure_reason",
            F.concat_ws(
                "; ",
                "dq_issues"
            )
        )
    )

    pass_count = passed.count()
    fail_count = failed.count()

    # --------------------------------------------------------
    # Reconciliation
    # --------------------------------------------------------

    assert (
        input_count
        ==
        pass_count + fail_count
    ), (
        f"DQ reconciliation failed for {system}"
    )

    print(
        f"PASS      : {pass_count:,}"
    )

    print(
        f"QUARANTINE: {fail_count:,}"
    )

    print(
        f"TOTAL     : {input_count:,}"
    )

    # --------------------------------------------------------
    # Write quarantine
    # --------------------------------------------------------

    (
        failed
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(
            f"{CATALOG}.quarantine.dq_{system}"
        )
    )

    # --------------------------------------------------------
    # Write trusted Silver
    # --------------------------------------------------------

    (
        passed
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(
            f"{CATALOG}.silver.{system}"
        )
    )

    print(
        f"Silver written: "
        f"{CATALOG}.silver.{system}"
    )

    return {
        "system": system,
        "input": input_count,
        "passed": pass_count,
        "quarantined": fail_count,
    }

###Run DQ for all sources

In [0]:
dq_results = []

for system in SYSTEMS:

    result = process_dq_for_source(
        system
    )

    dq_results.append(result)

dq_results

DQ PROCESSING: CORE
Input rows: 964,240
PASS      : 964,228
QUARANTINE: 12
TOTAL     : 964,240
Silver written: bfsi_mdm_dev.silver.core
DQ PROCESSING: CRM
Input rows: 660,223
PASS      : 660,214
QUARANTINE: 9
TOTAL     : 660,223
Silver written: bfsi_mdm_dev.silver.crm
DQ PROCESSING: KYC
Input rows: 811,890
PASS      : 811,890
QUARANTINE: 0
TOTAL     : 811,890
Silver written: bfsi_mdm_dev.silver.kyc
DQ PROCESSING: WEALTH
Input rows: 202,541
PASS      : 202,537
QUARANTINE: 4
TOTAL     : 202,541
Silver written: bfsi_mdm_dev.silver.wealth


[{'system': 'core', 'input': 964240, 'passed': 964228, 'quarantined': 12},
 {'system': 'crm', 'input': 660223, 'passed': 660214, 'quarantined': 9},
 {'system': 'kyc', 'input': 811890, 'passed': 811890, 'quarantined': 0},
 {'system': 'wealth', 'input': 202541, 'passed': 202537, 'quarantined': 4}]

### Duplicate `source_id` Integrity Metric

This is a **source-level Data Quality diagnostic**. It checks for repeated
non-blank `source_id` values within each source system before Entity Resolution.

This metric is intentionally separate from the DQ pass/quarantine logic:
duplicate source IDs are measured here, while cross-system customer matching
and entity resolution remain part of the MDM layer.

In [0]:
duplicate_source_id_metrics = []

for system in SYSTEMS:

    print("=" * 80)
    print(f"DUPLICATE SOURCE_ID METRIC: {system.upper()}")
    print("=" * 80)

    df = spark.table(
        f"{CATALOG}.silver.stg_{system}_standardized"
    )

    # Exclude blank source IDs because those are already handled
    # by the source identity integrity DQ check.
    non_blank_ids = df.filter(
        F.trim(F.col("source_id")) != ""
    )

    duplicate_groups = (
        non_blank_ids
        .groupBy("source_id")
        .count()
        .filter(F.col("count") > 1)
    )

    duplicate_source_id_count = duplicate_groups.count()

    duplicate_row_count = (
        duplicate_groups
        .agg(F.sum("count").alias("duplicate_rows"))
        .first()["duplicate_rows"]
        or 0
    )

    source_row_count = df.count()

    duplicate_source_id_metrics.append({
        "system": system,
        "source_rows": source_row_count,
        "duplicate_source_id_count": duplicate_source_id_count,
        "duplicate_row_count": int(duplicate_row_count),
    })

duplicate_source_id_metrics_df = spark.createDataFrame(
    duplicate_source_id_metrics
)

display(
    duplicate_source_id_metrics_df.orderBy("system")
)


DUPLICATE SOURCE_ID METRIC: CORE
DUPLICATE SOURCE_ID METRIC: CRM
DUPLICATE SOURCE_ID METRIC: KYC
DUPLICATE SOURCE_ID METRIC: WEALTH


duplicate_row_count,duplicate_source_id_count,source_rows,system
0,0,964240,core
0,0,660223,crm
0,0,811890,kyc
0,0,202541,wealth


### DQ Failure Reasons Check

In [0]:
for system in ["core", "crm", "wealth"]:

    print("=" * 80)
    print(f"DQ FAILURE REASONS: {system.upper()}")
    print("=" * 80)

    spark.table(
        f"bfsi_mdm_dev.quarantine.dq_{system}"
    ).groupBy(
        "dq_failure_reason"
    ).count().orderBy(
        F.desc("count")
    ).show(
        truncate=False
    )

DQ FAILURE REASONS: CORE
+-------------------------------------+-----+
|dq_failure_reason                    |count|
+-------------------------------------+-----+
|validity: normalized_email is invalid|12   |
+-------------------------------------+-----+

DQ FAILURE REASONS: CRM
+-------------------------------------+-----+
|dq_failure_reason                    |count|
+-------------------------------------+-----+
|validity: normalized_email is invalid|9    |
+-------------------------------------+-----+

DQ FAILURE REASONS: WEALTH
+-------------------------------------+-----+
|dq_failure_reason                    |count|
+-------------------------------------+-----+
|validity: normalized_email is invalid|4    |
+-------------------------------------+-----+



In [0]:
for system in ["core", "crm", "wealth"]:

    print("=" * 80)
    print(f"INVALID EMAILS: {system.upper()}")
    print("=" * 80)

    spark.table(
        f"bfsi_mdm_dev.quarantine.dq_{system}"
    ).select(
        "source_id",
        "name",
        "email",
        "normalized_email",
        "dq_failure_reason"
    ).show(
        50,
        truncate=False
    )

INVALID EMAILS: CORE
+---------+----------------------------------+------------------------------------------+------------------------------------------+-------------------------------------+
|source_id|name                              |email                                     |normalized_email                          |dq_failure_reason                    |
+---------+----------------------------------+------------------------------------------+------------------------------------------+-------------------------------------+
|C00892437|ULIVING@ESSEX3 LLP                |uliving@essex3.llp@example.com            |uliving@essex3.llp@example.com            |validity: normalized_email is invalid|
|C00895505|ULIVING@ESSEX2 ISSUERCO PLC       |uliving@essex2.issuerco.plc@example.com   |uliving@essex2.issuerco.plc@example.com   |validity: normalized_email is invalid|
|C00905859|@SIPP SIPP                        |@sipp.sipp@example.com                    |@sipp.sipp@example.com             

###Create unified Silver

In [0]:
silver_dfs = []

for system in SYSTEMS:

    df = spark.table(
        f"{CATALOG}.silver.{system}"
    )

    silver_dfs.append(df)


all_silver = silver_dfs[0]

for df in silver_dfs[1:]:

    all_silver = all_silver.unionByName(
        df
    )

###Write all_clients

In [0]:
(
    all_silver
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.silver.all_clients"
    )
)

print(
    "Written:",
    f"{CATALOG}.silver.all_clients"
)

Written: bfsi_mdm_dev.silver.all_clients


###Verify Silver

In [0]:
print(
    "Total Silver rows:",
    spark.table(
        f"{CATALOG}.silver.all_clients"
    ).count()
)

Total Silver rows: 2638869


In [0]:
display(
    spark.table(
        f"{CATALOG}.silver.all_clients"
    ).limit(20)
)

source_system,source_id,name,normalized_name,dob,normalized_dob,phone,normalized_phone,email,normalized_email,address,normalized_address,tax_id,lei,account_id,branch_id,segment,kyc_status,verification_date,normalized_verification_date,aum,risk_profile,relationship_manager
CORE,C00000001,Connor Smith,Connor Smith,1971-10-19,1971-10-19,+44(0)20 7496 0962,2074960962,elaine13@example.org,elaine13@example.org,"Studio 8 Hopkins park, West Scott, South East, RG16 2RP","Studio 8 Hopkins park, West Scott, South East, RG16 2RP",,,ACC000000001,BR001,,,,,null,,
CORE,C00000002,Kevin Ruth Hayes,Kevin Ruth Hayes,1970-05-26,1970-05-26,01632 960381,1632960381,jasminehopkins@example.net,jasminehopkins@example.net,"881 Davies hollow, Charlesborough, South East, SO6 2NK","881 Davies hollow, Charlesborough, South East, SO6 2NK",,,ACC000000002,BR002,,,,,null,,
CORE,C00000003,Jemma Dean,Jemma Dean,1957-03-03,1957-03-03,+44117 4960356,1174960356,lewisbrian@example.net,lewisbrian@example.net,"7 Donald trace, Wallacefurt, Scotland, PA5 2ZM","7 Donald trace, Wallacefurt, Scotland, PA5 2ZM",,,ACC000000003,BR003,,,,,null,,
CORE,C00000004,Richard Duncan Chambers,Richard Duncan Chambers,1983-10-21,1983-10-21,(0909) 879 0389,9098790389,walkergraham@example.org,walkergraham@example.org,"187 Fowler via, North Shaneborough, South East, TN17 7LC","187 Fowler via, North Shaneborough, South East, TN17 7LC",,,ACC000000004,BR004,,,,,null,,
CORE,C00000005,Harry Sharpe,Harry Sharpe,1967-09-22,1967-09-22,+44161 496 0148,1614960148,jenna56@example.org,jenna56@example.org,"Studio 5 Olivia motorway, Geraldineberg, East Midlands, NG8 6JP","Studio 5 Olivia motorway, Geraldineberg, East Midlands, NG8 6JP",,,ACC000000005,BR005,,,,,null,,
CORE,C00000006,Keith Francis Wilson,Keith Francis Wilson,1984-05-13,1984-05-13,(0141) 4960037,1414960037,ndavey@example.org,ndavey@example.org,"0 Welch spurs, Stewarttown, London, W12 6VP","0 Welch spurs, Stewarttown, London, W12 6VP",,,ACC000000006,BR006,,,,,null,,
CORE,C00000007,Mitchell Janice Blackburn,Mitchell Janice Blackburn,1974-06-16,1974-06-16,(0113)4960059,1134960059,darrendouglas@example.com,darrendouglas@example.com,"Studio 9 Simmons rapids, Port Ashley, Scotland, DD13 2GW","Studio 9 Simmons rapids, Port Ashley, Scotland, DD13 2GW",,,ACC000000007,BR007,,,,,null,,
CORE,C00000008,Raymond Marshall,Raymond Marshall,1988-06-26,1988-06-26,+4429 2018 0580,2920180580,dayraymond@example.net,dayraymond@example.net,"714 Yvonne manor, East Justin, North West, SK2 7HW","714 Yvonne manor, East Justin, North West, SK2 7HW",,,ACC000000008,BR008,,,,,null,,
CORE,C00000009,Leanne Robert Stevens,Leanne Robert Stevens,1982-04-11,1982-04-11,+44(0)9098790874,9098790874,robertscarol@example.com,robertscarol@example.com,"Studio 88J Sara wells, East Jade, London, WC16 8FG","Studio 88J Sara wells, East Jade, London, WC16 8FG",,,ACC000000009,BR009,,,,,null,,
CORE,C00000010,Dean Reynolds,Dean Reynolds,1978-02-07,1978-02-07,+44(0)1914960387,1914960387,damianrowe@example.net,damianrowe@example.net,"Flat 75H Ali union, Georgiachester, North West, CH20 7ZU","Flat 75H Ali union, Georgiachester, North West, CH20 7ZU",,,ACC000000011,BR011,,,,,null,,


In [0]:
spark.table(
    f"{CATALOG}.silver.all_clients"
).printSchema()

root
 |-- source_system: string (nullable = true)
 |-- source_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- normalized_name: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- normalized_dob: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- normalized_phone: string (nullable = true)
 |-- email: string (nullable = true)
 |-- normalized_email: string (nullable = true)
 |-- address: string (nullable = true)
 |-- normalized_address: string (nullable = true)
 |-- tax_id: string (nullable = true)
 |-- lei: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- branch_id: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- kyc_status: string (nullable = true)
 |-- verification_date: string (nullable = true)
 |-- normalized_verification_date: string (nullable = true)
 |-- aum: decimal(18,2) (nullable = true)
 |-- risk_profile: string (nullable = true)
 |-- relationship_manager: string (nullable = true)

